# Shopper Spectrum
Customer Segmentation & Product Recommendation

---
### Project Flow
1. Load & Understand Data
2. Data Preprocessing
3. EDA
4. RFM Feature Engineering
5. Customer Segmentation
6. Product Recommendation
7. Save Models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings, os, joblib
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
os.makedirs("models", exist_ok=True)
print("All libraries imported!")

## Step 1: Load Dataset

In [ ]:
df = pd.read_excel("data/online_retail.xlsx", engine="openpyxl")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})

In [ ]:
df.describe()

## Step 2: Data Preprocessing

In [ ]:
print("Records before cleaning:", len(df))

df = df.dropna(subset=["CustomerID"])
print("After removing missing CustomerID:", len(df))

df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
print("After removing cancelled invoices:", len(df))

df = df[df["Quantity"] > 0]
print("After removing zero/negative quantity:", len(df))

df = df[df["UnitPrice"] > 0]
print("After removing zero/negative price:", len(df))

df = df.drop_duplicates()
print("After removing duplicates:", len(df))

df["CustomerID"] = df["CustomerID"].astype(int).astype(str)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

print("Cleaning complete!")
df.head()

## Step 3: Exploratory Data Analysis

In [ ]:
country_counts = df["Country"].value_counts().head(10)
plt.figure(figsize=(12, 5))
sns.barplot(x=country_counts.values, y=country_counts.index, palette="Blues_r")
plt.title("Top 10 Countries by Transaction Volume", fontsize=14, fontweight="bold")
plt.xlabel("Number of Transactions")
plt.tight_layout()
plt.show()

In [ ]:
top_products = df.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(12, 5))
sns.barplot(x=top_products.values, y=top_products.index, palette="Greens_r")
plt.title("Top 10 Best-Selling Products", fontsize=14, fontweight="bold")
plt.xlabel("Total Quantity Sold")
plt.tight_layout()
plt.show()

In [ ]:
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M")
monthly_sales = df.groupby("YearMonth")["TotalPrice"].sum().reset_index()
monthly_sales["YearMonth"] = monthly_sales["YearMonth"].astype(str)
plt.figure(figsize=(14, 5))
plt.plot(monthly_sales["YearMonth"], monthly_sales["TotalPrice"], marker="o", color="steelblue", linewidth=2)
plt.title("Monthly Revenue Trend", fontsize=14, fontweight="bold")
plt.xlabel("Month"); plt.ylabel("Total Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
df_plot = df[df["TotalPrice"] < df["TotalPrice"].quantile(0.99)]
sns.histplot(df_plot["TotalPrice"], bins=60, color="coral", kde=True)
plt.title("Revenue Distribution per Transaction", fontsize=14, fontweight="bold")
plt.xlabel("Transaction Value")
plt.tight_layout()
plt.show()

## Step 4: RFM Feature Engineering

In [ ]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print("Snapshot Date:", snapshot_date)

rfm = df.groupby("CustomerID").agg(
    Recency   = ("InvoiceDate",  lambda x: (snapshot_date - x.max()).days),
    Frequency = ("InvoiceNo",    "nunique"),
    Monetary  = ("TotalPrice",   "sum")
).reset_index()

print("RFM Table Shape:", rfm.shape)
rfm.head(10)

In [ ]:
rfm[["Recency","Frequency","Monetary"]].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ["Recency","Frequency","Monetary"], ["steelblue","seagreen","coral"]):
    data = rfm[col]
    cap = data.quantile(0.99)
    sns.histplot(data[data <= cap], bins=40, ax=ax, color=color, kde=True)
    ax.set_title(f"{col} Distribution", fontweight="bold")
plt.tight_layout()
plt.show()

## Step 5: Customer Segmentation (KMeans Clustering)

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[["Recency", "Frequency", "Monetary"]])
print("Features scaled!")

In [ ]:
inertia = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertia, marker="o", color="steelblue", linewidth=2)
axes[0].set_title("Elbow Method - Inertia", fontweight="bold")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")
axes[1].plot(K_range, sil_scores, marker="s", color="seagreen", linewidth=2)
axes[1].set_title("Silhouette Score", fontweight="bold")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Score")
plt.tight_layout()
plt.show()
print("Best k by Silhouette Score:", K_range[sil_scores.index(max(sil_scores))])

In [ ]:
K = 4
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)
print("Clustering done!")
print(rfm["Cluster"].value_counts())

In [ ]:
cluster_summary = rfm.groupby("Cluster")[["Recency","Frequency","Monetary"]].mean().round(2)
print("Cluster RFM Averages:")
cluster_summary

In [ ]:
def label_cluster(row):
    r, f, m = row["Recency"], row["Frequency"], row["Monetary"]
    r_med = cluster_summary["Recency"].median()
    f_med = cluster_summary["Frequency"].median()
    m_med = cluster_summary["Monetary"].median()
    if r <= r_med and f >= f_med and m >= m_med:
        return "High-Value"
    elif r > r_med and f < f_med and m < m_med:
        if f == cluster_summary["Frequency"].min():
            return "At-Risk"
        return "Occasional"
    elif r <= r_med:
        return "Regular"
    else:
        return "At-Risk"

cluster_labels = cluster_summary.apply(label_cluster, axis=1).to_dict()
rfm["Segment"] = rfm["Cluster"].map(cluster_labels)
print("Segment Distribution:")
print(rfm["Segment"].value_counts())
rfm.head()

In [ ]:
fig = px.scatter(
    rfm, x="Recency", y="Monetary", color="Segment",
    size="Frequency", hover_data=["CustomerID"],
    title="Customer Segments - Recency vs Monetary",
    template="plotly_white"
)
fig.show()

In [ ]:
fig3d = px.scatter_3d(
    rfm, x="Recency", y="Frequency", z="Monetary",
    color="Segment", opacity=0.7,
    title="3D Customer Segments (RFM)",
    template="plotly_white"
)
fig3d.show()

In [ ]:
seg_counts = rfm["Segment"].value_counts().reset_index()
seg_counts.columns = ["Segment", "Count"]
fig_pie = px.pie(seg_counts, names="Segment", values="Count",
                 title="Customer Segment Distribution", template="plotly_white")
fig_pie.show()

## Step 6: Product Recommendation (Collaborative Filtering)

In [ ]:
top_n_products = df["Description"].value_counts().head(500).index
df_filtered = df[df["Description"].isin(top_n_products)]

customer_product = df_filtered.pivot_table(
    index="CustomerID",
    columns="Description",
    values="Quantity",
    aggfunc="sum",
    fill_value=0
)
print("Customer-Product Matrix Shape:", customer_product.shape)

In [ ]:
product_matrix = customer_product.T
similarity_matrix = cosine_similarity(product_matrix)
similarity_df = pd.DataFrame(similarity_matrix, index=product_matrix.index, columns=product_matrix.index)
print("Similarity Matrix Shape:", similarity_df.shape)
similarity_df.iloc[:5, :5]

In [ ]:
def get_recommendations(product_name, top_n=5):
    matches = [p for p in similarity_df.index if product_name.upper() in p.upper()]
    if not matches:
        return f"Product '{product_name}' not found."
    product = matches[0]
    similar = similarity_df[product].sort_values(ascending=False).drop(index=product)
    top = similar.head(top_n).reset_index()
    top.columns = ["Product", "Similarity Score"]
    top["Similarity Score"] = top["Similarity Score"].round(4)
    return top

result = get_recommendations("WHITE HANGING HEART", top_n=5)
print("Recommendations:")
print(result)

In [ ]:
top15 = df["Description"].value_counts().head(15).index
heatmap_data = similarity_df.loc[top15, top15]
plt.figure(figsize=(14, 10))
sns.heatmap(heatmap_data, annot=False, cmap="YlOrRd", linewidths=0.5)
plt.title("Product Similarity Heatmap (Top 15)", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

## Step 7: Save Models for Streamlit App

In [ ]:
joblib.dump(kmeans,         "models/kmeans_model.pkl")
joblib.dump(scaler,         "models/scaler.pkl")
joblib.dump(cluster_labels, "models/cluster_labels.pkl")
similarity_df.to_pickle(    "models/similarity_df.pkl")

print("All models saved to /models!")
print("Now run:  streamlit run app.py")